# TravelFit — Preprocessing Data Lengkap (Gabungan)

Cermin dari `01_preprocessing_travelfit.ipynb` (semua 7 tahap dipertahankan) dengan sumber **dataset gabungan 2.337 baris**: 1.900 sintetis 38 provinsi + 437 real Jawa. Tidak ada tahap yang dihapus; penyesuaian hanya pada sumber baca, pengisian harmonisasi, dan nama file keluaran (`*_full`).


In [1]:
import json
import re
import sys
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/TravelFit')
else:
    PROJECT_ROOT = Path.cwd().resolve()
    if PROJECT_ROOT.name.lower() == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports' / 'preprocessing'
for folder in (RAW_DIR, PROCESSED_DIR, REPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print('Environment :', 'Google Colab + Drive' if IN_COLAB else 'Local')
print('Project     :', PROJECT_ROOT)


Environment : Local
Project     : C:\Users\ASUS\Documents\SPK_Destinasi_Wisata


In [2]:
# Sumber: file Excel gabungan (lokal, bagian dari repo) + rating pengguna.
MERGED_XLSX = PROJECT_ROOT / 'Dataset_Wisata_Gabungan_2337.xlsx'
URLS = {
    'ratings': 'https://raw.githubusercontent.com/akhiyarwaladi/indonesia_tourism/main/tourism_rating.csv',
}
FILES = {
    'ratings': RAW_DIR / 'tourism_rating.csv',
}

for name, url in URLS.items():
    if not FILES[name].exists():
        print('Mengunduh:', FILES[name].name)
        urlretrieve(url, FILES[name])

destinations = pd.read_excel(MERGED_XLSX, sheet_name='Destinasi_Gabungan')
ratings = pd.read_csv(FILES['ratings'])
print('Destinasi gabungan:', destinations.shape, '| Rating:', ratings.shape)
print(destinations['source_dataset'].value_counts().to_string())
destinations.head(10)


Destinasi gabungan: (2337, 31) | Rating: (10000, 3)
source_dataset
synthetic_38_provinsi    1900
real_jawa_437             437


,place_id,place_id_original,source_dataset,place_name,description,category_original,sub_category,city,province,price,...,toilet,parking,food,worship,accessibility,information_center,c4_facility_score,activity_tags,feature_source_note,source_url
0,1,1,real_jawa_437,Monumen Nasional,Monumen Nasional atau yang populer disingkat d...,Budaya,NaN,Jakarta,DKI Jakarta,20000,...,False,False,False,False,False,False,0.0,budaya|sejarah,C4 dan activity_tags merupakan heuristik deskr...,https://github.com/akhiyarwaladi/indonesia_tou...
1,2,2,real_jawa_437,Kota Tua,"Kota tua di Jakarta, yang juga bernama Kota Tu...",Budaya,NaN,Jakarta,DKI Jakarta,0,...,False,False,False,False,False,False,0.0,budaya|edukasi|sejarah,C4 dan activity_tags merupakan heuristik deskr...,https://github.com/akhiyarwaladi/indonesia_tou...
2,3,3,real_jawa_437,Dunia Fantasi,Dunia Fantasi atau disebut juga Dufan adalah t...,Taman Hiburan,NaN,Jakarta,DKI Jakarta,270000,...,False,False,False,False,False,False,0.0,rekreasi_keluarga,C4 dan activity_tags merupakan heuristik deskr...,https://github.com/akhiyarwaladi/indonesia_tou...
3,4,4,real_jawa_437,Taman Mini Indonesia Indah (TMII),Taman Mini Indonesia Indah merupakan suatu kaw...,Taman Hiburan,NaN,Jakarta,DKI Jakarta,10000,...,False,False,False,False,False,False,0.0,budaya|rekreasi_keluarga,C4 dan activity_tags merupakan heuristik deskr...,https://github.com/akhiyarwaladi/indonesia_tou...
4,5,5,real_jawa_437,Atlantis Water Adventure,Atlantis Water Adventure atau dikenal dengan A...,Taman Hiburan,NaN,Jakarta,DKI Jakarta,94000,...,False,False,False,False,False,False,0.0,berenang|rekreasi_keluarga,C4 dan activity_tags merupakan heuristik deskr...,https://github.com/akhiyarwaladi/indonesia_tou...
5,6,6,real_jawa_437,Taman Impian Jaya Ancol,Taman Impian Jaya Ancol merupakan sebuah objek...,Taman Hiburan,NaN,Jakarta,DKI Jakarta,25000,...,False,False,False,False,False,False,0.0,rekreasi_keluarga,C4 dan activity_tags merupakan heuristik deskr...,https://github.com/akhiyarwaladi/indonesia_tou...
6,7,7,real_jawa_437,Kebun Binatang Ragunan,Kebun Binatang Ragunan adalah sebuah kebun bin...,Cagar Alam,NaN,Jakarta,DKI Jakarta,4000,...,False,False,False,False,False,False,0.0,belanja,C4 dan activity_tags merupakan heuristik deskr...,https://github.com/akhiyarwaladi/indonesia_tou...
7,8,8,real_jawa_437,Ocean Ecopark,Ocean Ecopark Salah satu zona rekreasi Ancol y...,Taman Hiburan,NaN,Jakarta,DKI Jakarta,180000,...,False,False,False,False,False,False,0.0,edukasi|rekreasi_keluarga,C4 dan activity_tags merupakan heuristik deskr...,https://github.com/akhiyarwaladi/indonesia_tou...
8,9,9,real_jawa_437,Pelabuhan Marina,Pelabuhan Marina Ancol berada di kawasan Taman...,Bahari,NaN,Jakarta,DKI Jakarta,175000,...,False,False,False,False,False,False,0.0,rekreasi_keluarga,C4 dan activity_tags merupakan heuristik deskr...,https://github.com/akhiyarwaladi/indonesia_tou...
9,10,10,real_jawa_437,Pulau Tidung,Pulau Tidung adalah salah satu kelurahan di ke...,Bahari,NaN,Jakarta,DKI Jakarta,150000,...,False,False,False,False,False,False,0.0,NaN,C4 dan activity_tags merupakan heuristik deskr...,https://github.com/akhiyarwaladi/indonesia_tou...


In [3]:
# 1. Bersihkan kolom, teks, tipe data, duplikat, dan baris tidak valid.
def snake_case(value):
    return re.sub(r'[^0-9a-zA-Z]+', '_', str(value).strip()).strip('_').lower()

destinations.columns = [snake_case(c) for c in destinations.columns]
ratings.columns = [snake_case(c) for c in ratings.columns]
destinations = destinations.loc[:, ~destinations.columns.str.startswith('unnamed')].copy()
ratings = ratings.loc[:, ~ratings.columns.str.startswith('unnamed')].copy()

for column in ['place_name', 'description', 'category_original', 'city']:
    destinations[column] = destinations[column].astype('string').str.replace(r'\s+', ' ', regex=True).str.strip()
for column in ['place_id', 'price', 'rating', 'time_minutes', 'lat', 'long']:
    destinations[column] = pd.to_numeric(destinations[column], errors='coerce')
for column in ['user_id', 'place_id', 'place_ratings']:
    ratings[column] = pd.to_numeric(ratings[column], errors='coerce')

destinations = destinations.drop_duplicates('place_id', keep='first')
destinations = destinations.loc[
    destinations['place_id'].notna()
    & destinations['place_name'].notna()
    & destinations['price'].ge(0)
    & destinations['rating'].between(1, 5)
    & destinations['lat'].between(-90, 90)
    & destinations['long'].between(-180, 180)
].copy()
print('Data valid:', destinations.shape)


Data valid: (2337, 31)


In [4]:
# 2. Seragamkan wilayah dan kategori (diisi hanya bila kosong; nilai gabungan dipertahankan).
CITY_TO_PROVINCE = {
    'Jakarta': 'DKI Jakarta', 'Bandung': 'Jawa Barat',
    'Semarang': 'Jawa Tengah', 'Yogyakarta': 'DI Yogyakarta',
    'Surabaya': 'Jawa Timur',
}
CATEGORY_MAP = {
    'budaya': 'budaya', 'taman hiburan': 'hiburan',
    'cagar alam': 'alam', 'pusat perbelanjaan': 'belanja',
    'tempat ibadah': 'religi', 'bahari': 'bahari',
}
# Kanonik provinsi mengikuti file gabungan (varian XLSX 38 provinsi).
PROVINCE_ALIAS = {'DI Yogyakarta': 'Daerah Istimewa Yogyakarta'}
destinations['province'] = (destinations['province'].fillna(destinations['city'].map(CITY_TO_PROVINCE))
    .replace(PROVINCE_ALIAS).astype('string'))
destinations['category_original'] = destinations['category_original'].astype('string').str.strip()
filled_category = destinations['category_original'].str.casefold().map(CATEGORY_MAP).astype('string')
destinations['category_clean'] = destinations['category_clean'].fillna(filled_category)
destinations[['category_original', 'category_clean']].drop_duplicates()


,category_original,category_clean
0,Budaya,budaya
2,Taman Hiburan,hiburan
6,Cagar Alam,alam
8,Bahari,bahari
14,Pusat Perbelanjaan,belanja
21,Tempat Ibadah,religi
437,Pantai,bahari
451,Gunung,alam


In [5]:
# 3. Bersihkan dan agregasikan rating pengguna (hanya baris real yang punya rating).
ratings = ratings.loc[
    ratings['user_id'].notna()
    & ratings['place_id'].notna()
    & ratings['place_ratings'].between(1, 5)
].drop_duplicates(['user_id', 'place_id'], keep='last')

rating_summary = ratings.groupby('place_id', as_index=False).agg(
    user_rating_mean=('place_ratings', 'mean'),
    user_rating_count=('place_ratings', 'count'),
    user_rating_std=('place_ratings', 'std'),
)
rating_summary['user_rating_std'] = rating_summary['user_rating_std'].fillna(0)
# Agregat file gabungan dihitung ulang dari file rating agar satu sumber kebenaran.
for column in ['user_rating_mean', 'user_rating_count', 'user_rating_std']:
    if column in destinations.columns:
        destinations = destinations.drop(columns=[column])
destinations = destinations.merge(rating_summary, on='place_id', how='left', validate='one_to_one')
destinations['c1_ticket_price'] = destinations['price'].astype(float)
destinations['c2_rating'] = destinations['rating'].astype(float)
destinations[['place_name', 'c1_ticket_price', 'c2_rating', 'user_rating_count']].head()


,place_name,c1_ticket_price,c2_rating,user_rating_count
0,Monumen Nasional,20000.0,4.6,18.0
1,Kota Tua,0.0,4.6,24.0
2,Dunia Fantasi,270000.0,4.6,18.0
3,Taman Mini Indonesia Indah (TMII),10000.0,4.5,21.0
4,Atlantis Water Adventure,94000.0,4.5,23.0


In [6]:
# 4. Bentuk fitur fasilitas C4 dan tag aktivitas untuk C6.
# Keduanya masih heuristik dari deskripsi dan perlu verifikasi untuk penelitian final.
# Pada dataset gabungan, heuristik hanya MENGISI sel kosong agar nilai kurasi tetap utuh.
FACILITY_KEYWORDS = {
    'toilet': ['toilet', 'kamar mandi', 'wc umum'],
    'parking': ['parkir', 'parking'],
    'food': ['warung', 'restoran', 'rumah makan', 'kafe', 'cafe', 'kuliner'],
    'worship': ['mushola', 'musala', 'masjid', 'tempat ibadah', 'gereja', 'pura', 'vihara'],
    'accessibility': ['disabilitas', 'difabel', 'kursi roda', 'wheelchair', 'aksesibel'],
    'information_center': ['pusat informasi', 'information center', 'layanan informasi'],
}
ACTIVITY_KEYWORDS = {
    'hiking': ['hiking', 'mendaki', 'pendakian', 'trekking'],
    'fotografi': ['fotografi', 'spot foto', 'berfoto', 'pemandangan', 'panorama'],
    'snorkeling': ['snorkeling', 'snorkel'],
    'diving': ['diving', 'menyelam', 'selam'],
    'camping': ['camping', 'berkemah', 'bumi perkemahan'],
    'kuliner': ['kuliner', 'makanan khas', 'jajanan', 'warung', 'restoran'],
    'sejarah': ['sejarah', 'bersejarah', 'peninggalan', 'museum', 'monumen'],
    'budaya': ['budaya', 'tradisi', 'kesenian', 'keraton'],
    'belanja': ['belanja', 'pusat perbelanjaan', 'pasar', 'mal', 'mall'],
    'berenang': ['berenang', 'kolam renang', 'waterpark', 'water park'],
    'edukasi': ['edukasi', 'pendidikan', 'belajar', 'museum'],
    'religi': ['ziarah', 'religi', 'ibadah', 'masjid', 'gereja', 'pura', 'vihara'],
    'rekreasi_keluarga': ['keluarga', 'wahana', 'taman bermain', 'taman hiburan'],
}
CATEGORY_TAGS = {
    'budaya': {'budaya'}, 'belanja': {'belanja'},
    'religi': {'religi'}, 'hiburan': {'rekreasi_keluarga'},
}

def normalize_text(value):
    return re.sub(r'\s+', ' ', str(value).casefold()).strip() if pd.notna(value) else ''

def contains_keyword(text, keyword):
    escaped = re.escape(keyword.casefold()).replace(r'\ ', r'\s+')
    return re.search(rf'(?<!\w){escaped}(?!\w)', text) is not None

description_text = destinations['description'].map(normalize_text)
facility_columns = []
for facility, keywords in FACILITY_KEYWORDS.items():
    column = f'facility_{facility}_mentioned'
    heuristic = description_text.map(
        lambda text, keys=keywords: int(any(contains_keyword(text, key) for key in keys))
    )
    if column in destinations.columns:
        destinations[column] = destinations[column].fillna(heuristic).astype(int)
    else:
        destinations[column] = heuristic
    facility_columns.append(column)
need_c4 = destinations['c4_facility_score'].isna()
destinations.loc[need_c4, 'c4_facility_score'] = destinations.loc[need_c4, facility_columns].mean(axis=1)

def extract_activity_tags(row):
    text = normalize_text(f"{row['place_name']} {row['description']}")
    tags = set(CATEGORY_TAGS.get(row['category_clean'], set()))
    for tag, keywords in ACTIVITY_KEYWORDS.items():
        if any(contains_keyword(text, keyword) for keyword in keywords):
            tags.add(tag)
    return '|'.join(sorted(tags))

need_tags = destinations['activity_tags'].isna() | (destinations['activity_tags'] == '')
filled = destinations.loc[need_tags].apply(extract_activity_tags, axis=1)
destinations.loc[need_tags, 'activity_tags'] = filled.values
destinations.loc[need_tags, 'feature_source_note'] = 'C4 dan activity_tags merupakan heuristik deskripsi; perlu verifikasi'
print('Baris diisi heuristik:', int(need_tags.sum()))
destinations[['place_name', 'c4_facility_score', 'category_clean', 'activity_tags']].head(10)


Baris diisi heuristik: 71


,place_name,c4_facility_score,category_clean,activity_tags
0,Monumen Nasional,0.0,budaya,budaya|sejarah
1,Kota Tua,0.0,budaya,budaya|edukasi|sejarah
2,Dunia Fantasi,0.0,hiburan,rekreasi_keluarga
3,Taman Mini Indonesia Indah (TMII),0.0,hiburan,budaya|rekreasi_keluarga
4,Atlantis Water Adventure,0.0,hiburan,berenang|rekreasi_keluarga
5,Taman Impian Jaya Ancol,0.0,hiburan,rekreasi_keluarga
6,Kebun Binatang Ragunan,0.0,alam,belanja
7,Ocean Ecopark,0.0,hiburan,edukasi|rekreasi_keluarga
8,Pelabuhan Marina,0.0,bahari,rekreasi_keluarga
9,Pulau Tidung,0.0,bahari,


In [7]:
# 5. Standardisasi fitur numerik dan one-hot encoding kategori untuk K-Means

NUMERIC_FEATURES = [
    "c1_ticket_price",
    "c2_rating",
    "c4_facility_score",
]

numeric_data = destinations[NUMERIC_FEATURES].astype(float)

# Standardisasi Z-score
feature_mean = numeric_data.mean()
feature_scale = numeric_data.std(ddof=0).replace(0, 1)

scaled_data = (numeric_data - feature_mean) / feature_scale
scaled_data.columns = [
    f"{column}_z" for column in NUMERIC_FEATURES
]

# One-hot encoding kategori
category_data = pd.get_dummies(
    destinations["category_clean"],
    prefix="category",
    dtype=int,
)

# Gabungkan identitas destinasi dengan fitur hasil transformasi
kmeans_ready = pd.concat(
    [
        destinations[
            [
                "place_id",
                "place_name",
                "city",
                "province",
                "category_clean",
            ]
        ].reset_index(drop=True),
        scaled_data.reset_index(drop=True),
        category_data.reset_index(drop=True),
    ],
    axis=1,
)

print("Dataset bersih:", destinations.shape)
print("Siap K-Means :", kmeans_ready.shape)

kmeans_ready.head()


Dataset bersih: (2337, 37)
Siap K-Means : (2337, 14)


,place_id,place_name,city,province,category_clean,c1_ticket_price_z,c2_rating_z,c4_facility_score_z,category_alam,category_bahari,category_belanja,category_budaya,category_hiburan,category_religi
0,1,Monumen Nasional,Jakarta,DKI Jakarta,budaya,-0.106340,0.329440,-2.027074,0,0,0,1,0,0
1,2,Kota Tua,Jakarta,DKI Jakarta,budaya,-0.593244,0.329440,-2.027074,0,0,0,1,0,0
2,3,Dunia Fantasi,Jakarta,DKI Jakarta,hiburan,5.979968,0.329440,-2.027074,0,0,0,0,1,0
3,4,Taman Mini Indonesia Indah (TMII),Jakarta,DKI Jakarta,hiburan,-0.349792,-0.111259,-2.027074,0,0,0,0,1,0
4,5,Atlantis Water Adventure,Jakarta,DKI Jakarta,hiburan,1.695207,-0.111259,-2.027074,0,0,0,0,1,0


In [8]:
# 6. Simpan hasil preprocessing (varian _full; file 01 tidak ditimpa).
destinations['source_dataset'] = destinations.get('source_dataset', 'gabungan')
destinations['source_url'] = 'Dataset_Wisata_Gabungan_2337.xlsx'

destinations_path = PROCESSED_DIR / 'destinations_full_clean.csv'
kmeans_path = PROCESSED_DIR / 'destinations_full_kmeans_ready.csv'
ratings_path = PROCESSED_DIR / 'ratings_aggregated_full.csv'
summary_path = REPORT_DIR / 'preprocessing_summary_full.json'

destinations.to_csv(destinations_path, index=False, encoding='utf-8-sig')
kmeans_ready.to_csv(kmeans_path, index=False, encoding='utf-8-sig')
rating_summary.to_csv(ratings_path, index=False, encoding='utf-8-sig')

summary = {
    'destination_rows': int(len(destinations)),
    'rating_rows_valid': int(len(ratings)),
    'cities': sorted(destinations['city'].dropna().unique().tolist()),
    'categories': sorted(destinations['category_clean'].dropna().unique().tolist()),
    'destinations_with_facility_evidence': int(destinations['c4_facility_score'].gt(0).sum()),
    'destinations_with_activity_tags': int(destinations['activity_tags'].ne('').sum()),
    'scaler_mean': feature_mean.to_dict(),
    'scaler_scale': feature_scale.to_dict(),
}
with open(summary_path, 'w', encoding='utf-8') as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)

assert destinations['place_id'].is_unique
assert destinations['c1_ticket_price'].ge(0).all()
assert destinations['c2_rating'].between(1, 5).all()
assert destinations['c4_facility_score'].between(0, 1).all()
assert not kmeans_ready.isna().any().any()

print('Preprocessing selesai.')
print('-', destinations_path)
print('-', kmeans_path)
print('-', ratings_path)
print('-', summary_path)


Preprocessing selesai.
- C:\Users\ASUS\Documents\SPK_Destinasi_Wisata\data\processed\destinations_full_clean.csv
- C:\Users\ASUS\Documents\SPK_Destinasi_Wisata\data\processed\destinations_full_kmeans_ready.csv
- C:\Users\ASUS\Documents\SPK_Destinasi_Wisata\data\processed\ratings_aggregated_full.csv
- C:\Users\ASUS\Documents\SPK_Destinasi_Wisata\reports\preprocessing\preprocessing_summary_full.json


## Output

- `destinations_full_clean.csv`: data gabungan bersih dan fitur awal (2.337 baris).
- `destinations_full_kmeans_ready.csv`: data terstandardisasi untuk K-Means.
- `ratings_aggregated_full.csv`: ringkasan rating per destinasi.
- `preprocessing_summary_full.json`: ringkasan preprocessing.

C3 jarak, C5 kecocokan kategori, dan C6 Jaccard pengguna dihitung pada tahap modeling setelah data survei tersedia.
